# Experiment v3 on a Jupyter server

Run the v3 EMCTS-crossover experiment on a big remote box (this was written for ~1 TB RAM / 128 cores).

**Prerequisites**
1. Upload (or `git clone`) the whole repo to the server, including the `src/` and `lib/` folders.
2. A JDK (Java 8+) and Python 3 with pip are available. (`!java -version`, `!python --version`.)
3. Run this notebook **from the repo's `Tribes-master/` directory** (set the path in the next cell).

Notes
- This box is Linux, so the classpath separator is `:` (Windows uses `;`).
- The full run is long (the 5000 ms matchups dominate). It is **resumable** - re-running `RunExperimentV3` skips finished work - so use `nohup` + a log (below) rather than a blocking cell, in case the browser/kernel disconnects.
- Use one server (the user limited themselves to one). 120 threads leaves headroom for the OS.

In [ ]:
import os
# EDIT this to the absolute path of the Tribes-master directory on the server:
REPO = os.path.expanduser('~/Bachelor_Thesis/Tribes-master')
os.chdir(REPO)
print('cwd =', os.getcwd())
print(os.listdir('.'))   # expect to see src/, lib/, analysis/ ...

In [ ]:
# 0. Environment check
!java -version
!python --version
!nproc                 # number of cores
!free -g | head -2     # RAM in GB

## 1. Compile
Compiles all `src/**/*.java` into `out/`, with the bundled jars in `lib/` on the classpath.

In [ ]:
!rm -rf out && mkdir -p out
!find src -name '*.java' > _srcs.txt
!javac -d out -cp "lib/*" @_srcs.txt && echo 'BUILD OK' || echo 'BUILD FAILED'

## 2. (Pre-step) Re-tune OEP at 5000 ms
v3 Phase 2 includes the 5000 ms level, so OEP needs 5000 ms params. This sweeps population x tournament
and writes `test_results/v3/oep_tuning/oep_best_params.txt`. **Then** edit the 5000 ms branch of
`ExpAgents.oepTime()` with the printed `BEST_POP`/`BEST_TOURNAMENT`, recompile (cell 1), and continue.

**Keep POP_SIZE &lt;= 100.** Much larger populations (e.g. 200) can stall OEP when its population converges
to fewer distinct individuals than the tournament size in the early game - and the optimum is small anyway
(tuning has found smaller populations better at larger budgets: 500ms->100, 1000ms->50), so {20,50,100}
covers the useful range. (If you only care about 500/1000 ms, you can skip this - the placeholder is fine.)

In [ ]:
!java -Xmx64g -cp "out:lib/*" core.data.exp.OepTuning 120 5000 20,50,100 3,5,10 10 test_results/v3/oep_tuning
!echo '---'; cat test_results/v3/oep_tuning/oep_best_params.txt

## 3. Run v3 (background + log)
Starts the full run detached so it survives a disconnect. Monitor with the `tail` cell (re-run to refresh).
Set `-Xmx` to most of RAM (e.g. 700g of 1 TB) and the thread count to ~120.

In [ ]:
!nohup java -Xmx700g -cp "out:lib/*" core.data.exp.RunExperimentV3 120 > v3_run.log 2>&1 &
print('v3 started in the background -> v3_run.log')

In [ ]:
# Monitor progress (re-run this cell to refresh). Also: is the java process still alive?
!tail -n 40 v3_run.log
!echo '--- java processes ---'; pgrep -af RunExperimentV3 || echo '(none - finished or not started)'

## 4. Analyse (after the run completes)
v3 is a **separate** experiment (fixed turn-3 start, no MCTS), so analyse it on its own. Output goes to
`analysis/figures_v3/` and `analysis/tables_v3/`.

In [ ]:
!python -m pip install -q pandas matplotlib numpy
!python analysis/analyze.py test_results/v3

In [ ]:
# Show the figures inline
import glob
from IPython.display import Image, display
for f in sorted(glob.glob('analysis/figures_v3/*.png')):
    print(f); display(Image(filename=f))